# IoT Hub Monitor — обучение тяжёлых ML-моделей в Google Colab (Фаза 4, инкр. 2)

Пайплайн: **БД → CSV → Colab (обучение) → веса → мак → локальный инференс**.

Этот ноутбук **импортирует те же `ml/`-модули, что и прод** (через `git clone` публичной
репы) — он НЕ дублирует логику моделей и persistence. Обученный артефакт
(`<key>.manifest.json` + веса) идентичен локальному, потому что используется тот же
`ModelPersistenceMixin.save`.

**Что нужно заранее:** CSV, выгруженный командой
`python manage.py export_training_data --device TEMP-001 --metric temperature --output train.csv`.

**Важно про воспроизводимость:** учим на **CPU** — GPU даёт другой порядок операций и,
следовательно, другие веса. Сети крошечные (автоэнкодер ~800 параметров), на CPU Colab
это секунды. GPU оставляем как ускорение для будущих больших моделей.

## 1. Клонируем репу и добавляем её корень в `sys.path`

`iot_hub/__init__.py` отсутствует намеренно — пакет резолвится как implicit namespace
package (Python 3), ровно как в `tests/`. Достаточно положить корень репы в `sys.path`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/vadim-white/IoT-hub-monitor.git"
REPO_DIR = "IoT-hub-monitor"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL

repo_root = os.path.abspath(REPO_DIR)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("repo_root:", repo_root)

## 2. Фиксируем версии зависимостей

Версии **из `requirements.txt`** — критично для воспроизводимости весов. Colab несёт свои
torch/numpy; без пина веса могут чуть отличаться. Django НЕ ставим — `ml/`-ядро от него
не зависит (`loader.py` с ORM в Colab не используем, ряд собираем из CSV).

In [ ]:
# torch CPU-only — детерминизм важнее скорости (см. шапку)
!pip install -q torch==2.2.2 numpy==1.26.3 scikit-learn==1.4.2
# раскомментировать, если учим Holt-Winters forecaster:
# !pip install -q statsmodels==0.14.2

## 3. Загружаем CSV и собираем `TimeSeries`

`load_series_from_csv` — офлайн-зеркало `loader.load_series` (тот же `TimeSeries`,
те же метки). Загрузите CSV через виджет ниже (или смонтируйте Google Drive).

In [ ]:
from google.colab import files
uploaded = files.upload()          # выберите train.csv из export_training_data
CSV_PATH = next(iter(uploaded))

# альтернатива — Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/train.csv'
print("CSV:", CSV_PATH)

In [ ]:
from iot_hub.apps.telemetry.ml.dataset_csv import load_series_from_csv

DEVICE = "TEMP-001"        # serial_number, как в колонке CSV
METRIC = "temperature"     # metric_type

series = load_series_from_csv(CSV_PATH, DEVICE, METRIC)
print(f"точек: {len(series)}, аномалий: {int(series.labels.sum())}")

## 4. Обучаем модель

Гиперпараметры собираем через `cli_params` — тот же источник, что у команд
`train_models`/`detect_anomalies`. Это гарантирует, что **sha манифеста совпадёт** и
локальный `detect_anomalies --use-cache` примет веса без `CacheMismatchError`.

Меняйте `METHOD`/`opts` под нужную модель (`autoencoder` детектор, `lstm` форкастер).

In [ ]:
from iot_hub.apps.telemetry.ml.detectors import build_detector
from iot_hub.apps.telemetry.ml.cli_params import detector_params
import time

METHOD = "autoencoder"
# те же дефолты, что у train_models — менять ЗДЕСЬ, потом теми же значениями звать --use-cache
opts = {
    "window": 24, "threshold": 3.0,
    "epochs": 150, "latent_dim": 8, "lr": 1e-3,
    "threshold_percentile": 98.0, "random_state": 42,
    "contamination": 0.02, "n_estimators": 200,
}

model = build_detector(METHOD, **detector_params(METHOD, opts))
t0 = time.perf_counter()
model.fit(series)
print(f"{model.name} обучен за {time.perf_counter() - t0:.2f}с")

## 5. Сохраняем артефакт через тот же `save()`, что и прод

In [ ]:
from iot_hub.apps.telemetry.ml.persistence import model_key

stem = model_key(model.name, DEVICE, METRIC)   # ml/models/<name>__<device>__<metric>
model.save(stem)
print("артефакт:", stem.name)
!ls -la {stem.parent}

## 6. Скачиваем веса

Распакуйте архив локально в `iot_hub/apps/telemetry/ml/models/`, затем:
`python manage.py detect_anomalies --device TEMP-001 --metric temperature \`
`--method autoencoder --use-cache` — команда загрузит веса вместо переобучения.

In [ ]:
from google.colab import files

models_dir = stem.parent
!cd {repo_root}/iot_hub/apps/telemetry/ml && zip -r /content/ml_models.zip models
files.download("/content/ml_models.zip")